# Phase 1 : OFFLINE & CALIBRATION

In [1]:
import os
from datasets import load_dataset, load_from_disk
import pandas as pd
from IPython.display import display, Markdown

from encoders import build_encoder
from utils_indexation import build_indices
import config

/home/marion/MIRAGE_TFE/env_marion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
ENCODER_REGISTRY = {
    "BLIP":      (build_encoder("blip"),      256),
    "SigLIP":    (build_encoder("siglip"),    1152),
    "DINO_BERT": (build_encoder("dino_bert"), 512),
    "CONVNEXT_CLIP":(build_encoder("convnext_clip"), 1024),
    "EVA_CLIP":   (build_encoder("eva_clip"),  768),
}

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 616/616 [00:00<00:00, 2262.91it/s, Materializing param=vision_proj.weight]                                                 
BlipForImageTextRetrieval LOAD REPORT from: Salesforce/blip-itm-large-coco
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_encoder.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is

Chargement de ConvNeXt-CLIP (XXLarge)...
Chargement de EVA-CLIP...
-> Modèle sélectionné : EVA02-L-14 | Poids : merged2b_s4b_b131k


In [3]:
DATASETS_CONFIG = {
    "flickr30k": {
        "hf_path": "nlphuji/flickr30k",
        "custom_split": True,
        "image_col": "image",
        "caption_col": "caption"
    },
    "flickr8k": {
        "hf_path": "dandelin/flickr8k",
        "custom_split": False,
        "image_col": "image",
        "caption_col": "captions"
    },
}

config_ds = DATASETS_CONFIG[config.DATASET_NAME]

## Chargement des Données

In [4]:
print(f"Chargement du dataset : {config.DATASET_NAME}...")

if os.path.exists(f"{config.RAW_DATA_DIR}/train"):
    print("Chargement depuis le disque local...")
    train_dataset = load_from_disk(f"{config.RAW_DATA_DIR}/train")
    test_dataset  = load_from_disk(f"{config.RAW_DATA_DIR}/test")
    val_dataset   = load_from_disk(f"{config.RAW_DATA_DIR}/val")
else:
    print("Téléchargement et formatage depuis HuggingFace...")
    if config_ds["custom_split"]:
        # Logique spécifique (ex: Flickr30k de nlphuji)
        raw_dataset = load_dataset(config_ds["hf_path"], trust_remote_code=True)['test']
        train_dataset = raw_dataset.filter(lambda x: x['split'] == 'train')
        test_dataset  = raw_dataset.filter(lambda x: x['split'] == 'test')
        val_dataset   = raw_dataset.filter(lambda x: x['split'] == 'val')
    else:
        # Logique standard HuggingFace (ex: Flickr8k)
        raw_dataset = load_dataset(config_ds["hf_path"], trust_remote_code=True)
        train_dataset = raw_dataset['train']
        test_dataset  = raw_dataset['test']
        # Gérer le nom du split de validation (parfois 'validation', parfois 'val')
        val_split_name = 'validation' if 'validation' in raw_dataset else 'val'
        val_dataset   = raw_dataset[val_split_name]
    
    # Sauvegarde pour la prochaine fois
    train_dataset.save_to_disk(f"{config.RAW_DATA_DIR}/train")
    test_dataset.save_to_disk(f"{config.RAW_DATA_DIR}/test")
    val_dataset.save_to_disk(f"{config.RAW_DATA_DIR}/val")

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Chargement du dataset : flickr30k...
Chargement depuis le disque local...
Train: 29000, Val: 1014, Test: 1000


## Calibration

In [5]:
needs_calibration = any(encoder.needs_calibration for encoder, _ in ENCODER_REGISTRY.values())

if needs_calibration:
    cal_images, cal_texts = [], []
    
    for i, item in enumerate(train_dataset):
        if len(cal_texts) >= 10000: break
        captions = item[config_ds['caption_col']] if isinstance(item[config_ds['caption_col']], list) else [item[config_ds['caption_col']]]
        for cap in captions:
            cal_images.append(item[config_ds['image_col']])
            cal_texts.append(cap)

    print(f"Paires prêtes : {len(cal_texts)}")
    
    for name, (encoder, _) in ENCODER_REGISTRY.items():
        if encoder.needs_calibration:
            encoder.calibrate(cal_images, cal_texts, batch_size=32, epochs=20)
            
    print("\nCalibration terminée avec succès !")

Paires prêtes : 10000
1. Extraction des vecteurs bruts...
2. Entraînement contrastif stable...
  Époque 01/20 | Loss: 3.6916
  Époque 02/20 | Loss: 1.7124
  Époque 03/20 | Loss: 1.1930
  Époque 04/20 | Loss: 0.9363
  Époque 05/20 | Loss: 0.7730
  Époque 06/20 | Loss: 0.6640
  Époque 07/20 | Loss: 0.5842
  Époque 08/20 | Loss: 0.5188
  Époque 09/20 | Loss: 0.4696
  Époque 10/20 | Loss: 0.4314
  Époque 11/20 | Loss: 0.3994
  Époque 12/20 | Loss: 0.3738
  Époque 13/20 | Loss: 0.3527
  Époque 14/20 | Loss: 0.3344
  Époque 15/20 | Loss: 0.3278
  Époque 16/20 | Loss: 0.3102
  Époque 17/20 | Loss: 0.2967
  Époque 18/20 | Loss: 0.2820
  Époque 19/20 | Loss: 0.2821
  Époque 20/20 | Loss: 0.2746

Calibration terminée avec succès !


## Indexation et Sauvegarde

In [6]:
print("\n--- Indexation Validation ---")
val_corpus = build_indices(
    val_dataset,
    ENCODER_REGISTRY,
    image_field   = config_ds['image_col'],
    caption_field = config_ds['caption_col'],
    batch_size    = 32,
    save_dir      = config.INDEX_DIR,
    prefix        = "val_",
    store_vectors = True,
)

print("\n--- Indexation Test ---")
test_corpus = build_indices(
    test_dataset,
    ENCODER_REGISTRY,
    image_field   = config_ds['image_col'],
    caption_field = config_ds['caption_col'],
    batch_size    = 64,
    save_dir      = config.INDEX_DIR,
    prefix        = "test_",
    store_vectors = True,
)


--- Indexation Validation ---

[build_indices | val_]
  1014 images | 5070 textes (5 caption(s)/image)


val_textes: 100%|██████████| 159/159 [00:11<00:00, 13.80it/s]


  [BLIP] img=1014 | txt=5070  ✓
  [SigLIP] img=1014 | txt=5070  ✓
  [DINO_BERT] img=1014 | txt=5070  ✓
  [CONVNEXT_CLIP] img=1014 | txt=5070  ✓
  [EVA_CLIP] img=1014 | txt=5070  ✓
  Sauvegarde dans ./data/flickr30k/index_sauvegardes...
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_blip_img_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_blip_txt_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_siglip_img_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_siglip_txt_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_dino_bert_img_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_dino_bert_txt_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_convnext_clip_img_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_convnext_clip_txt_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/val_eva_clip_img_index.bin + .pkl
  Sauveg

test_textes: 100%|██████████| 79/79 [00:11<00:00,  7.17it/s]


  [BLIP] img=1000 | txt=5000  ✓
  [SigLIP] img=1000 | txt=5000  ✓
  [DINO_BERT] img=1000 | txt=5000  ✓
  [CONVNEXT_CLIP] img=1000 | txt=5000  ✓
  [EVA_CLIP] img=1000 | txt=5000  ✓
  Sauvegarde dans ./data/flickr30k/index_sauvegardes...
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_blip_img_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_blip_txt_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_siglip_img_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_siglip_txt_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_dino_bert_img_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_dino_bert_txt_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_convnext_clip_img_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_convnext_clip_txt_index.bin + .pkl
  Sauvegardé : ./data/flickr30k/index_sauvegardes/test_eva_clip_img_index.bin + .pkl

## Résultats des Temps d'Indexation (Val + Test)

In [7]:
total_timing = {}
for name in ENCODER_REGISTRY:
    total_timing[name] = {
        "Images (s)": test_corpus.timing_stats[name]["Images (s)"],
        "Textes (s)": test_corpus.timing_stats[name]["Textes (s)"],
        "Total (s)": test_corpus.timing_stats[name]["Total (s)"]
    }

# Création du DataFrame avec les nouveaux temps
df_new_times = pd.DataFrame.from_dict(total_timing, orient='index')

# 2. Chargement de l'historique et Fusion
if os.path.exists(config.TEMPS_INDEX_CSV):
    # On charge les anciens temps
    df_existing = pd.read_csv(config.TEMPS_INDEX_CSV, index_col=0)
    # On met à jour avec les nouveaux temps (s'il y a des doublons, on garde le nouveau)
    df_times = pd.concat([df_existing, df_new_times])
    df_times = df_times[~df_times.index.duplicated(keep='last')]
else:
    # C'est la première fois qu'on lance le code
    df_times = df_new_times

# 3. Formatage pour le Markdown (On met en gras la valeur la plus BASSE)
df_formatted_times = df_times.copy()
for col in df_formatted_times.columns:
    min_val = df_times[col].min() # Cherche le minimum parmi tous les modèles
    df_formatted_times[col] = df_times[col].apply(
        lambda x: f"**{x:.2f}**" if x == min_val else f"{x:.2f}"
    )

# 4. Affichage
print("\n" + "="*60)
print("RÉSULTATS - TEMPS D'INDEXATION EN SECONDES (Set de Test uniquement)")
print("="*60)
display(Markdown(df_formatted_times.to_markdown()))

# 5. Sauvegarde
df_times.to_csv(config.TEMPS_INDEX_CSV, float_format="%.2f")
df_formatted_times.to_markdown(config.TEMPS_INDEX_FILE)

print(f"\n✅ Temps d'indexation (Test) mis à jour avec succès dans :")
print(f"   - {config.TEMPS_INDEX_CSV}")
print(f"   - {config.TEMPS_INDEX_FILE}")


RÉSULTATS - TEMPS D'INDEXATION EN SECONDES (Set de Test uniquement)


|               | Images (s)   | Textes (s)   | Total (s)   |
|:--------------|:-------------|:-------------|:------------|
| BLIP          | 8.64         | 1.38         | 10.02       |
| SigLIP        | 10.72        | 3.68         | 14.40       |
| DINO_BERT     | **1.33**     | **0.88**     | **2.21**    |
| CONVNEXT_CLIP | 9.88         | 3.68         | 13.57       |
| EVA_CLIP      | 4.24         | 1.33         | 5.58        |


✅ Temps d'indexation (Test) mis à jour avec succès dans :
   - ./data/flickr30k/index_sauvegardes/temps_indexation.csv
   - ./data/flickr30k/index_sauvegardes/temps_indexation.md
